# R6 · Exponentielle et logarithmes — notebook d'exercices

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookMaths/blob/main/chapitres/r6_exp_log/03_notebook.ipynb)

La partie 0 vérifie tes exercices papier. Chaque exercice de code : un énoncé, une cellule à compléter (les `...` et les `raise NotImplementedError`), puis une cellule de vérification (`wbm.check`). « Tout exécuter » (*Run all*) va jusqu'au bout même si rien n'est rempli : ce qui n'est pas fait affiche ⏳. Bloqué 15 minutes ? `04_indices.md`.

> Travaille dans **ta copie** (`mon_travail/r6_exp_log/03_notebook.ipynb`, créée par `python tools/start_chapter.py R6`) ; la version du dépôt est mise à jour par Claude. Sur Colab, le badge ouvre la version du dépôt, qui n'est pas enregistrée : crée puis ouvre ta copie comme l'explique `00_setup/COLAB.md` §2.

| Partie | ID | Titre | Type | ★ | ⏱️ |
|---|---|---|---|---|---|
| 0 | R6.1 à R6.8, R6.11, R6.12 | Les exercices papier (vérification) | ✏️ 🔗 | | |
| A | R6.14 | Un logarithme dans n'importe quelle base · `log_base` | 🔨 | ★ | 15 |
|  | R6.15 | Sigmoïde et tanh à partir de exp · `sigmoid`, `tanh` | 🔨 | ★★ | 20 |
|  | R6.16 | Temps de doublement · `doubling_time` | 🔨 | ★ | 15 |
| B | R6.17 | Sur tout un tableau : `np.exp`, `np.log2` et `scipy.special.expit` | 📦 | ★ | 15 |
|  | R6.18 | Prédire : exp(800) avec math et NumPy, et le produit de 600 probabilités de 0,25 | 🔮 | ★★ | 15 |
|  | R6.19 | Croissances comparées : x¹⁰, 2ˣ et ln x de plus en plus loin | 🔬 | ★★ | 20 |
|  | R6.20 | Les mêmes données en échelle linéaire et en échelle logarithmique | 🎨 | ★ | 15 |
| C | R6.21 | Chasse à l'erreur : ln(a + b), e^(a + b) et log(0) | 🐛 | ★★ | 15 |
|  | R6.22 | Lire la documentation de `numpy.log`, `log2`, `log10`, `log1p` | 🛠️ | ★ | 15 |

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (a few seconds per cell); False = bigger experiments
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookMaths.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookMaths")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WBM_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wbm").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wbm").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier de workbookMaths.")

os.environ["WBM_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wbm  # noqa: E402

cfg = wbm.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R6")  # your own library

## Objectifs et rappel express

- Vérifier tes calculs papier (partie 0) : propriétés de exp et de ln, équations, changements de base, bits et nats, doublements, sigmoïde et logit, échelles logarithmiques, log-probabilités.
- Écrire les quatre fonctions de `mymath/explog.py` et les tester.
- Calculer sur des tableaux avec NumPy, comparer des croissances par leurs logarithmes, tracer une échelle logarithmique, chasser des règles mal appliquées et lire une documentation.

**Rappel express.** $e^{a+b} = e^a e^b$, $(e^a)^b = e^{ab}$, $e^x > 0$. $\ln$ est la réciproque de exp : $\ln(e^y) = y$, $e^{\ln x} = x$ pour $x > 0$. $\ln(ab) = \ln a + \ln b$, $\ln(a^k) = k \ln a$ ; $\ln(a + b)$ ne se simplifie pas. $\log_b x = \frac{\ln x}{\ln b}$. $\sigma(x) = \frac{1}{1 + e^{-x}}$, $\sigma(-x) = 1 - \sigma(x)$, $\operatorname{logit}(p) = \ln\frac{p}{1 - p}$. `np.log` est $\ln$.

## Partie 0 · Vérifier tes exercices papier (✏️ R6.1 à R6.8, 🔗 R6.11 et R6.12)

Fais d'abord les exercices de `02_exercices.md` **sur papier**, dans ta copie de `06_mes_reponses.md`. Reporte ensuite chaque réponse ici : **la valeur** que tu as trouvée, sous la forme indiquée en commentaire, jamais le calcul qui y mène :
- un nombre avec un point décimal, `-1.5`, arrondi comme le demande la question ;
- une fraction exacte entre guillemets, `"5/7"` ;
- une expression entre guillemets, `"3*x - 1"` (`*` pour multiplier, `**` pour la puissance) ;
- un intervalle entre guillemets, `"[-2 ; 4["` (`-oo` et `+oo` pour l'infini) ;
- une liste de nombres entre crochets ; vrai ou faux par `True` ou `False` ; un numéro de réponse proposée par un entier.

Les réponses pas encore remplies affichent ⏳.

**Ex R6.1 — Simplifier avec les propriétés de exp**

In [ ]:
# Ex R6.1: the values you found on paper
answer_R6_1a = ...  # a) a number
answer_R6_1b = ...  # b) a number
answer_R6_1c = ...  # c) an expression in x, as text
answer_R6_1d = ...  # d) an expression in x, as text
answer_R6_1e = ...  # e) an integer
answer_R6_1f = ...  # f) a fraction, as text
answer_R6_1g = ...  # g) an expression in x, as text
answer_R6_1h = ...  # h) an integer

for letter, answer in zip("abcdefgh", [answer_R6_1a, answer_R6_1b, answer_R6_1c, answer_R6_1d, answer_R6_1e, answer_R6_1f, answer_R6_1g, answer_R6_1h]):
    wbm.check(f"R6.1{letter}", answer)

**Ex R6.2 — Simplifier avec les propriétés de ln**

In [ ]:
# Ex R6.2: the values you found on paper
answer_R6_2a = ...  # a) an integer
answer_R6_2b = ...  # b) an integer
answer_R6_2c = ...  # c) an integer
answer_R6_2d = ...  # d) a fraction, as text
answer_R6_2e = ...  # e) an integer
answer_R6_2f = ...  # f) an integer
answer_R6_2g = ...  # g) an expression in a and b, as text
answer_R6_2h = ...  # h) an expression in a and b, as text

for letter, answer in zip("abcdefgh", [answer_R6_2a, answer_R6_2b, answer_R6_2c, answer_R6_2d, answer_R6_2e, answer_R6_2f, answer_R6_2g, answer_R6_2h]):
    wbm.check(f"R6.2{letter}", answer)

**Ex R6.3 — Équations et inéquations avec exp et ln**

In [ ]:
# Ex R6.3: the values you found on paper
answer_R6_3a = ...  # a) a number (2 decimals)
answer_R6_3b = ...  # b) a number (2 decimals)
answer_R6_3c = ...  # c) a number (2 decimals)
answer_R6_3d = ...  # d) a number
answer_R6_3e = ...  # e) an integer
answer_R6_3f = ...  # f) an integer
answer_R6_3g = ...  # g) an interval, as text
answer_R6_3h = ...  # h) a list (4 decimals)

for letter, answer in zip("abcdefgh", [answer_R6_3a, answer_R6_3b, answer_R6_3c, answer_R6_3d, answer_R6_3e, answer_R6_3f, answer_R6_3g, answer_R6_3h]):
    wbm.check(f"R6.3{letter}", answer)

**Ex R6.4 — Changer de base : log₂, log₁₀, ln**

In [ ]:
# Ex R6.4: the values you found on paper
answer_R6_4a = ...  # a) an integer
answer_R6_4b = ...  # b) an integer
answer_R6_4c = ...  # c) an integer
answer_R6_4d = ...  # d) an integer
answer_R6_4e = ...  # e) a number (3 decimals)
answer_R6_4f = ...  # f) a number (3 decimals)
answer_R6_4g = ...  # g) an integer
answer_R6_4h = ...  # h) a fraction, as text

for letter, answer in zip("abcdefgh", [answer_R6_4a, answer_R6_4b, answer_R6_4c, answer_R6_4d, answer_R6_4e, answer_R6_4f, answer_R6_4g, answer_R6_4h]):
    wbm.check(f"R6.4{letter}", answer)

**Ex R6.5 — Compter en bits et en nats : combien de questions oui ou non ?**

In [ ]:
# Ex R6.5: the values you found on paper
answer_R6_5a = ...  # a) an integer
answer_R6_5b = ...  # b) a number (2 decimals)
answer_R6_5c = ...  # c) an integer
answer_R6_5d = ...  # d) a number (3 decimals)
answer_R6_5e = ...  # e) a number (2 decimals)
answer_R6_5f = ...  # f) a number (2 decimals)
answer_R6_5g = ...  # g) a number (4 decimals)
answer_R6_5h = ...  # h) a number (2 decimals)

for letter, answer in zip("abcdefgh", [answer_R6_5a, answer_R6_5b, answer_R6_5c, answer_R6_5d, answer_R6_5e, answer_R6_5f, answer_R6_5g, answer_R6_5h]):
    wbm.check(f"R6.5{letter}", answer)

**Ex R6.6 — Temps de doublement et demi-vie**

In [ ]:
# Ex R6.6: the values you found on paper
answer_R6_6a = ...  # a) a number (2 decimals)
answer_R6_6b = ...  # b) a number (2 decimals)
answer_R6_6c = ...  # c) a number (2 decimals)
answer_R6_6d = ...  # d) a number (4 decimals)
answer_R6_6e = ...  # e) a number (2 decimals)
answer_R6_6f = ...  # f) a number (2 decimals)
answer_R6_6g = ...  # g) a fraction, as text

for letter, answer in zip("abcdefg", [answer_R6_6a, answer_R6_6b, answer_R6_6c, answer_R6_6d, answer_R6_6e, answer_R6_6f, answer_R6_6g]):
    wbm.check(f"R6.6{letter}", answer)

**Ex R6.7 — Sigmoïde et logit : la symétrie σ(−x) = 1 − σ(x), la cote et la log-cote**

In [ ]:
# Ex R6.7: the values you found on paper
answer_R6_7a = ...  # a) a fraction, as text
answer_R6_7b = ...  # b) a fraction, as text
answer_R6_7c = ...  # c) a number (3 decimals)
answer_R6_7d = ...  # d) a number (3 decimals)
answer_R6_7e = ...  # e) a number
answer_R6_7f = ...  # f) a number (3 decimals)
answer_R6_7g = ...  # g) a number (3 decimals)
answer_R6_7h = ...  # h) a number

for letter, answer in zip("abcdefgh", [answer_R6_7a, answer_R6_7b, answer_R6_7c, answer_R6_7d, answer_R6_7e, answer_R6_7f, answer_R6_7g, answer_R6_7h]):
    wbm.check(f"R6.7{letter}", answer)

**Ex R6.8 — Placer des valeurs sur une échelle logarithmique**

In [ ]:
# Ex R6.8: the values you found on paper
answer_R6_8a = ...  # a) a number
answer_R6_8b = ...  # b) a number (2 decimals)
answer_R6_8c = ...  # c) a number (2 decimals)
answer_R6_8d = ...  # d) a number (2 decimals)
answer_R6_8e = ...  # e) a number (2 decimals)

for letter, answer in zip("abcde", [answer_R6_8a, answer_R6_8b, answer_R6_8c, answer_R6_8d, answer_R6_8e]):
    wbm.check(f"R6.8{letter}", answer)

**Ex R6.11 — Des produits de probabilités aux sommes de logarithmes**

In [ ]:
# Ex R6.11: the values you found on paper
answer_R6_11a = ...  # a) a number (4 decimals)
answer_R6_11b = ...  # b) a number (4 decimals)
answer_R6_11c = ...  # c) a number (4 decimals)
answer_R6_11d = ...  # d) a number (4 decimals)
answer_R6_11e = ...  # e) 1 or 2
answer_R6_11f = ...  # f) a number (2 decimals)
answer_R6_11g = ...  # g) True or False
answer_R6_11h = ...  # h) a number (1 decimal)

for letter, answer in zip("abcdefgh", [answer_R6_11a, answer_R6_11b, answer_R6_11c, answer_R6_11d, answer_R6_11e, answer_R6_11f, answer_R6_11g, answer_R6_11h]):
    wbm.check(f"R6.11{letter}", answer)

**Ex R6.12 — Des logarithmes au travail : TF-IDF et PSNR**

In [ ]:
# Ex R6.12: the values you found on paper
answer_R6_12a = ...  # a) a number
answer_R6_12b = ...  # b) a number (3 decimals)
answer_R6_12c = ...  # c) a number (3 decimals)
answer_R6_12d = ...  # d) a list (3 decimals)
answer_R6_12e = ...  # e) a number
answer_R6_12f = ...  # f) a number (2 decimals)
answer_R6_12g = ...  # g) a number (4 decimals)

for letter, answer in zip("abcdefg", [answer_R6_12a, answer_R6_12b, answer_R6_12c, answer_R6_12d, answer_R6_12e, answer_R6_12f, answer_R6_12g]):
    wbm.check(f"R6.12{letter}", answer)

## Partie A · Ta librairie : logarithmes, sigmoïde, tanh, doublement

Fiche §1, §3, §4 et §7. Les quatre fonctions de `mymath/explog.py`.

In [ ]:
# Tools of the notebook (parts A to C)
import math
import warnings

import matplotlib.pyplot as plt
import numpy as np

from wbm.nbtools import checked, drawn, run_pytest_group


def raised(function, *args):
    """The name of the exception that function(*args) raises, or "aucune" if it returns.

    NotImplementedError (a TODO not done yet) goes through, for wbm.attempt to show ⏳."""
    try:
        function(*args)
    except NotImplementedError:
        raise
    except Exception as exc:  # noqa: BLE001 - any other error is the answer
        return type(exc).__name__
    return "aucune"

### Ex R6.14 — Un logarithme dans n'importe quelle base · `log_base` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le logarithme en base quelconque avec le seul logarithme népérien, et refuser les entrées interdites.  
**Prérequis :** fiche §3 et §4 · **mymath :** `explog.py` · **Parcours :** C, E

> **Mode d'emploi `mymath`** (comme dans les chapitres précédents). Ouvre ta copie `mon_travail/mymath/explog.py`, lis la docstring de la fonction en entier, remplace la ligne `raise NotImplementedError(...)` par ton code, **enregistre** le fichier, puis exécute les cellules ci-dessous : la première recharge ta librairie. Tu n'as droit qu'à `math.exp` et `math.log` (le logarithme népérien, avec un seul argument) : un test le vérifie. Les autres tests comparent tes résultats à `math.log2`, `math.log10`, NumPy et SciPy.

Écris `log_base(x, base)` dans `mon_travail/mymath/explog.py` : le logarithme de `x` en base `base`, avec la formule de changement de base (fiche §4) et `math.log`. Une `ValueError` si `x <= 0`, si `base <= 0` ou si `base == 1` : vérifie-le **avant** de calculer.

La vérification calcule :
- a) $\log_{5} 15\,625$ et $\log_{2} 0{,}001\,953\,125$ ;
- b) $\log_{0{,}5} 10$ ;
- c) le nom de l'erreur levée pour $x = 0$ (avec `raised`, de la cellule des outils), puis d) pour la base 1.

Puis, dans ta copie : pourquoi la base 1 est-elle interdite ? Que donnerait la formule ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R6")  # reload your saved file

In [ ]:
with checked("R6.14a"):
    wbm.check("R6.14a", [mymath.explog.log_base(15625, 5), mymath.explog.log_base(0.001953125, 2)], computed=True)
with checked("R6.14b"):
    wbm.check("R6.14b", mymath.explog.log_base(10, 0.5), computed=True)
with checked("R6.14c"):
    wbm.check("R6.14c", raised(mymath.explog.log_base, 0, 10), computed=True)
with checked("R6.14d"):
    wbm.check("R6.14d", raised(mymath.explog.log_base, 5, 1), computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r6_explog.py", "test_log_base_ or test_explog_uses")

### Ex R6.15 — Sigmoïde et tanh à partir de exp · `sigmoid`, `tanh` 🔨 ★★ ⏱️ 20 min
**Objectif :** programmer la sigmoïde et la tangente hyperbolique avec la seule exponentielle, et vérifier leurs propriétés.  
**Prérequis :** fiche §7 ; R6.10 · **mymath :** `explog.py` · **Parcours :** C, E

Dans `mon_travail/mymath/explog.py` :
- `sigmoid(x)` : $\sigma(x) = \frac{1}{1 + e^{-x}}$, avec `math.exp` ;
- `tanh(x)` : $\tanh x = \frac{e^{x} - e^{-x}}{e^{x} + e^{-x}}$, avec `math.exp` (ou avec ta sigmoïde : fiche §7). `math.tanh` est interdite.

Les tests restent dans $\lvert x \rvert \le 30$ (la docstring) : pas besoin de précautions pour des $x$ énormes, que workbookIA prendra au chapitre 17.

La vérification calcule :
- a) $\sigma(0)$, $\sigma(1{,}5)$ et $\sigma(-1{,}5)$ ;
- b) si $\sigma(-30) > 0$ et si $\sigma(30) < 1$ (une liste de deux booléens) ;
- c) $\tanh 0$, $\tanh 0{,}75$ et $\tanh(-2)$ ;
- d) si $\tanh x = 2\,\sigma(2x) - 1$ à $10^{-12}$ près pour 101 valeurs de $x$ entre −5 et 5 ;
- e) $\tanh 30$ et $\tanh(-30)$.

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R6")  # reload your saved file

In [ ]:
with checked("R6.15"):
    sigmoid, tanh = mymath.explog.sigmoid, mymath.explog.tanh
    wbm.check("R6.15a", [sigmoid(0), sigmoid(1.5), sigmoid(-1.5)], computed=True)
    wbm.check("R6.15b", [bool(sigmoid(-30) > 0), bool(sigmoid(30) < 1)], computed=True)
    wbm.check("R6.15c", [tanh(0), tanh(0.75), tanh(-2)], computed=True)
    wbm.check("R6.15d", all(abs(tanh(x) - (2 * sigmoid(2 * x) - 1)) < 1e-12 for x in np.linspace(-5, 5, 101)),
              computed=True)
    wbm.check("R6.15e", [tanh(30), tanh(-30)], computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r6_explog.py", "test_sigmoid_ or test_tanh_ or test_explog_uses")

### Ex R6.16 — Temps de doublement · `doubling_time` 🔨 ★ ⏱️ 15 min
**Objectif :** programmer le temps de doublement d'une croissance à taux constant, et le comparer à la règle de 70.  
**Prérequis :** fiche §1 et §3 ; R6.6 · **mymath :** `explog.py` · **Parcours :** C

Écris `doubling_time(rate)` : le nombre **réel** $n$ de périodes tel que $(1 + \text{rate})^n = 2$ (fiche §3), sans l'arrondir. Une `ValueError` si `rate <= 0` (une grandeur qui ne grandit pas ne double jamais).

La vérification calcule :
- a) les temps de doublement à +7 % et à +25 % par période ;
- b) le nom de l'erreur levée pour un taux nul ;
- c) les temps de doublement à +1 %, +10 % et +50 %.

Puis, dans ta copie : compare les réponses de c) à la règle de 70 ($\frac{70}{\text{taux en \%}}$). Pour quels taux la règle est-elle bonne, et pourquoi (fiche §3) ?

In [ ]:
mymath = wbm.load_mymath("learner", missing_ok=True, fallback="ref", chapter="R6")  # reload your saved file

In [ ]:
with checked("R6.16a"):
    wbm.check("R6.16a", [mymath.explog.doubling_time(0.07), mymath.explog.doubling_time(0.25)], computed=True)
with checked("R6.16b"):
    wbm.check("R6.16b", raised(mymath.explog.doubling_time, 0), computed=True)
with checked("R6.16c"):
    wbm.check("R6.16c", [mymath.explog.doubling_time(r) for r in (0.01, 0.1, 0.5)], computed=True)

**Les tests** (lis d'abord la ligne qui suit chaque ❌ : elle dit ce qui était attendu).

In [ ]:
run_pytest_group("tests/test_r6_explog.py", "test_doubling_time_")

## Partie B · NumPy, une prédiction et des expériences

Fiche §3 à §7. Les mêmes fonctions sur tout un tableau, ce que font Python et NumPy d'un nombre trop grand ou trop petit, des croissances comparées par leurs logarithmes, puis deux échelles en image.

### Ex R6.17 — Sur tout un tableau : `np.exp`, `np.log2` et `scipy.special.expit` 📦 ★ ⏱️ 15 min
**Objectif :** appliquer exp, les logarithmes et la sigmoïde à tout un tableau d'un coup, et fabriquer une grille logarithmique.  
**Prérequis :** fiche §4, §6 et §7 ; workbookIA 0A §100.8.3 (calcul vectorisé) · **Parcours :** C

NumPy applique une fonction à chaque élément d'un tableau, sans boucle (0A §100.8.3) : `np.exp`, `np.log` (le logarithme népérien), `np.log2`, `np.log10`, `np.floor` (la partie entière, vers le bas) ; `scipy.special.expit` est la sigmoïde de SciPy. Et `np.logspace(a, b, n)` fabrique $n$ valeurs de $10^a$ à $10^b$, également espacées **sur une échelle logarithmique** (fiche §6). Calcule, chacun **en une ligne** :
- a) $\log_{2}$ de chaque nombre de `values_a` ;
- b) le nombre de chiffres de chaque entier de `values_b` (fiche §4) ;
- c) la sigmoïde de chaque nombre de `values_c` ;
- d) cinq valeurs de $10^{-4}$ à $10^{0}$, également espacées sur une échelle logarithmique.

In [ ]:
from scipy.special import expit

values_a = np.array([0.25, 2, 2048])
values_b = np.array([7, 4321, 1000000])
values_c = np.array([-2.5, 0, 2.5])

answer_R6_17a = ...  # a) an array of 3 numbers
answer_R6_17b = ...  # b) an array of 3 numbers of digits
answer_R6_17c = ...  # c) an array of 3 numbers
answer_R6_17d = ...  # d) an array of 5 numbers

In [ ]:
for letter, answer in zip("abcd", [answer_R6_17a, answer_R6_17b, answer_R6_17c, answer_R6_17d]):
    wbm.check(f"R6.17{letter}", answer)

### Ex R6.18 — Prédire : exp(800) avec math et NumPy, et le produit de 600 probabilités de 0,25 🔮 ★★ ⏱️ 15 min
**Objectif :** prévoir ce que font Python et NumPy d'un nombre trop grand ou trop petit pour un flottant, et voir pourquoi on calcule avec des log-probabilités.  
**Prérequis :** fiche §3 et §5 · **Parcours :** C, E

$e^{800}$ dépasse de loin le plus grand `float64` (≈ $1{,}8 \times 10^{308}$, fiche §5), et $0{,}25^{600} \approx 10^{-361}$ est bien plus petit que le plus petit flottant positif. Que font alors Python et NumPy ? **Avant d'exécuter quoi que ce soit**, prédis, pour chaque calcul, le numéro de ce qu'il donne :
- a) `math.exp(800)` : 1 un nombre fini immense ; 2 la valeur spéciale `inf` (l'infini des flottants) ; 3 une erreur qui arrête le calcul ;
- b) `np.exp(800.0)` : 1, 2 ou 3, comme en a) ;
- c) `np.prod(probabilities)`, le produit de 600 probabilités égales à 0,25 : 1 un tout petit nombre positif ; 2 exactement `0.0` ; 3 une erreur ;
- d) `np.log(probabilities).sum()`, la somme de leurs 600 logarithmes : 1 un nombre ordinaire ; 2 la valeur spéciale `-inf` ; 3 une erreur.

Écris tes quatre prédictions, vérifie-les, puis lance l'expérience : elle affiche chaque résultat, ou l'erreur levée, et les alertes émises.

📝 **Mon hypothèse** (à écrire **avant** d'exécuter quoi que ce soit) : …

In [ ]:
prediction_R6_18a = ...  # a) 1, 2 or 3, written BEFORE running the experiment below
prediction_R6_18b = ...  # b) 1, 2 or 3
prediction_R6_18c = ...  # c) 1, 2 or 3
prediction_R6_18d = ...  # d) 1, 2 or 3

In [ ]:
for letter, prediction in zip("abcd", [prediction_R6_18a, prediction_R6_18b, prediction_R6_18c,
                                          prediction_R6_18d]):
    wbm.check(f"R6.18{letter}", prediction)

**L'expérience.** Elle ne s'exécute qu'une fois tes quatre prédictions écrites.

In [ ]:
if any(answer is ... or answer is None for answer in [prediction_R6_18a, prediction_R6_18b, prediction_R6_18c, prediction_R6_18d]):
    print("⏳ Écris d'abord tes quatre prédictions, puis exécute à nouveau.")
else:
    def outcome(compute):
        """What compute() gives: its value, the name of the error it raises, and the warnings it emits."""
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            try:
                value = compute()
            except Exception as exc:  # noqa: BLE001 - the error is part of the answer
                value = f"erreur {type(exc).__name__} : {exc}"
        alerts = "; ".join(f"{type(w.message).__name__} : {w.message}" for w in caught)
        return value, alerts or "aucune alerte"


    probabilities = np.full(600, 0.25)
    for label, compute in [("a) math.exp(800)", lambda: math.exp(800)),
                           ("b) np.exp(800)", lambda: np.exp(800.0)),
                           ("c) np.prod(probabilities)", lambda: np.prod(probabilities)),
                           ("d) np.log(probabilities).sum()", lambda: np.log(probabilities).sum())]:
        value, alerts = outcome(compute)
        print(f"{label:32} -> {value}   [{alerts}]")

**Après l'expérience.** e) La valeur de la somme d), à 1 décimale : vérifie-la par un calcul à la main, avec $\ln 0{,}25$. Puis, dans ta copie : pourquoi, pour comparer deux modèles sur des milliers de données, calcule-t-on la somme des logarithmes des probabilités plutôt que leur produit ?

In [ ]:
answer_R6_18e = ...  # e) a number (1 decimal)

In [ ]:
wbm.check("R6.18e", answer_R6_18e)

### Ex R6.19 — Croissances comparées : x¹⁰, 2ˣ et ln x de plus en plus loin 🔬 ★★ ⏱️ 20 min
**Objectif :** comparer des fonctions trop grandes pour un flottant en comparant leurs logarithmes, et voir où une croissance en dépasse une autre.  
**Prérequis :** fiche §3 et §5 ; R6.18 · **Parcours :** C

$2^{1000}$ et $1000^{10}$ sont des nombres énormes, mais on peut les comparer **par leurs logarithmes**, qui restent petits (fiche §5) : $\ln(2^{n}) = n \ln 2$ et $\ln(n^{10}) = 10 \ln n$.

1. Calcule `gap`, le tableau des $\ln \frac{2^{n}}{n^{10}} = n \ln 2 - 10 \ln n$ pour chaque entier $n$ de `n` (de 2 à 1 000), en une ligne de NumPy. Puis :
   - a) combien d'entiers $n$ de 2 à 1 000 vérifient $2^{n} \le n^{10}$ ?
   - b) le plus petit entier $n$ à partir duquel $2^{n} > n^{10}$ pour tous les entiers suivants jusqu'à 1 000 ;
   - c) de combien d'ordres de grandeur $2^{1000}$ dépasse-t-il $1000^{10}$ ? (le $\log_{10}$ du rapport, 2 décimales)
2. Compare maintenant $x^{0{,}1}$ et $\ln x$, avec $t = \ln x$ : $\ln(x^{0{,}1}) = 0{,}1\,t$ et $\ln(\ln x) = \ln t$. Calcule `race`, le tableau des $0{,}1\,t - \ln t$ pour chaque $t$ de `t` (de 2 à 100). Puis :
   - d) le plus petit entier $t$ à partir duquel $x^{0{,}1} > \ln x$, c'est-à-dire $0{,}1\,t > \ln t$ ;
   - e) le plus petit entier $k$ tel que $\ln k > 15$ (pense à $e^{15}$).

Dans ta copie : à partir de quel ordre de grandeur de $x$ la puissance $x^{0{,}1}$ dépasse-t-elle enfin $\ln x$ ? Qu'en conclus-tu sur « finit par » ?

In [ ]:
n = np.arange(2, 1001)                                  # the integers 2, 3, …, 1000
t = np.arange(2, 101)                                   # values of ln x, from 2 to 100

In [ ]:
gap = ...            # 1. an array: n ln 2 − 10 ln n, for each n of n
race = ...           # 2. an array: 0.1 t − ln t, for each t of t
answer_R6_19a = ...  # a) an integer
answer_R6_19b = ...  # b) an integer
answer_R6_19c = ...  # c) a number (2 decimals)
answer_R6_19d = ...  # d) an integer
answer_R6_19e = ...  # e) an integer

In [ ]:
for letter, answer in zip("abcde", [answer_R6_19a, answer_R6_19b, answer_R6_19c, answer_R6_19d,
                                       answer_R6_19e]):
    wbm.check(f"R6.19{letter}", answer)

**La figure** : les deux écarts, en fonction de $n$ et de $t$.

In [ ]:
with checked("R6.19 figure"):
    if gap is ... or race is ...:
        print("⏳ Écris d'abord gap et race.")
    else:
        fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
        axes[0].plot(n, np.asarray(gap) / math.log(10), color="C0")
        axes[0].axhline(0, color="gray", linewidth=0.8)
        axes[0].set_xlabel("n")
        axes[0].set_title("log₁₀(2ⁿ / n¹⁰) : négatif, puis positif pour toujours")
        axes[1].plot(t, race, color="C1")
        axes[1].axhline(0, color="gray", linewidth=0.8)
        axes[1].set_xlabel("t = ln x")
        axes[1].set_title("ln(x^0,1) − ln(ln x) = 0,1 t − ln t")
        for ax in axes:
            ax.grid(True, alpha=0.3)
        fig.tight_layout()
        plt.show()

### Ex R6.20 — Les mêmes données en échelle linéaire et en échelle logarithmique 🎨 ★ ⏱️ 15 min
**Objectif :** tracer les mêmes données sur deux échelles, et lire une croissance exponentielle sur un graphique semi-logarithmique.  
**Prérequis :** fiche §6 ; R0 (matplotlib) · **Parcours :** C, E

Un comptage double à peu près tous les 1,5 pas : `counts` contient $12 \times 1{,}6^{t}$ pour $t = 0$ à 15 (`steps`). Complète `plot_scales(axes, steps, counts)`, qui reçoit **deux axes** côte à côte :
- dans `axes[0]`, les points `(steps, counts)` en échelle ordinaire ;
- dans `axes[1]`, les mêmes points, avec l'axe vertical en échelle logarithmique (`axes[1].set_yscale("log")`, fiche §6) ;
- un titre pour chacun, et des points plutôt qu'une ligne (`"o"`).

La cellule de vérification crée les axes, puis la figure s'affiche. Puis, en lisant le graphique de droite **et** en le justifiant par le calcul :
- a) de combien de décades la droite monte-t-elle à chaque pas ? ($\log_{10} 1{,}6$, 3 décimales)
- b) combien de pas faut-il pour monter d'une décade, c'est-à-dire multiplier le comptage par 10 ? (2 décimales)

Dans ta copie : sur lequel des deux graphiques vois-tu le mieux les premières valeurs ? Et si la croissance ralentissait à la fin, à quoi le verrais-tu sur le graphique de droite ?

In [ ]:
steps = np.arange(0, 16)
counts = 12 * 1.6**steps

In [ ]:
def plot_scales(axes, steps, counts):
    """axes[0]: the points (steps, counts) on a linear scale; axes[1]: the same points with a log y axis. Titles."""
    raise NotImplementedError("trace les deux graphiques dans axes[0] et axes[1]")


answer_R6_20a = ...  # a) a number (3 decimals)
answer_R6_20b = ...  # b) a number (2 decimals)

In [ ]:
with checked("R6.20 figure"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    try:
        created = drawn(plot_scales, axes, steps, counts)
    except BaseException:
        plt.close(fig)                                     # no empty figure while plot_scales is not written yet
        raise
    if created:
        plt.close(fig)
        print("❌ Ex R6.20 : ta fonction ouvre sa propre figure (plt.figure ou plt.subplots) : dessine dans les axes "
              "qu'elle reçoit, avec axes[0].plot(...) et axes[1].plot(...).")
    else:
        fig.tight_layout()
        plt.show()
for letter, answer in zip("ab", [answer_R6_20a, answer_R6_20b]):
    wbm.check(f"R6.20{letter}", answer)

**Vérifications automatiques** (la figure elle-même se juge à l'œil).

In [ ]:
with checked("R6.20 vérifications"):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    try:
        created = drawn(plot_scales, axes, steps, counts)
    finally:
        plt.close(fig)                                     # only the checks below are shown here
    problems = []
    if created:
        problems.append("ta fonction ouvre sa propre figure : dessine dans les axes reçus (axes[0].plot, …)")
    else:
        for k, ax in enumerate(axes):
            ys = []
            for line in ax.lines:
                ys += list(np.asarray(line.get_ydata(), dtype=float))
            for collection in ax.collections:
                ys += list(np.asarray(collection.get_offsets(), dtype=float)[:, 1])
            if not any(np.isclose(y, counts[-1]) for y in ys):
                problems.append(f"rien n'est tracé dans axes[{k}] (dessine les données avec axes[{k}].plot(...), pas "
                                f"plt.plot)")
        if axes[0].get_yscale() != "linear":
            problems.append("axes[0] doit garder l'échelle linéaire")
        if axes[1].get_yscale() != "log":
            problems.append("l'axe vertical de axes[1] en échelle logarithmique (axes[1].set_yscale(\"log\"))")
        if not all(ax.get_title() for ax in axes):
            problems.append("un titre pour chaque graphique (ax.set_title)")
    if problems:
        print("❌ Ex R6.20 : à revoir : " + " ; ".join(problems) + ".")
    else:
        print("✅ Ex R6.20 : deux graphiques, l'un en échelle linéaire, l'autre en échelle logarithmique.")

## Partie C · Erreurs de règles et documentation

Fiche §2 à §4. Trois règles mal appliquées à trouver, puis la documentation des logarithmes de NumPy.

### Ex R6.21 — Chasse à l'erreur : ln(a + b), e^(a + b) et log(0) 🐛 ★★ ⏱️ 15 min
**Objectif :** repérer les règles de exp et de ln mal appliquées dans du code, et ce que donne le logarithme de 0.  
**Prérequis :** fiche §2 et §3 · **Parcours :** C, E

Trois petites fonctions utilisent exp et ln. Certaines lignes sont fausses. **Sans rien exécuter d'abord**, trouve-les.

```python
def total_log_prob(probabilities):
    """ln of the product of independent probabilities, computed as a sum of logarithms."""
    total = 1                                                     # (A)
    for p in probabilities:                                       # (B)
        total = total + math.log(p)                               # (C)
    return total                                                  # (D)


def log_of_sum(a, b):
    """ln(a + b), for a > 0 and b > 0."""
    return math.log(a) + math.log(b)                              # (E)


def grow(start, rate, t1, t2):
    """start × e^(rate × t1), then that value × e^(rate × t2): the value after t1 + t2 of continuous growth."""
    return start * (math.exp(rate * t1) + math.exp(rate * t2))    # (F)
```

- a) les lettres des lignes fausses, dans l'ordre alphabétique (un texte) ;
- b) même une fois corrigée, `total_log_prob([0.5, 0.0])` ne rend pas de nombre : quel est le nom de l'erreur levée ? (`math.log` de 0 ; fiche §3)

Puis corrige les trois fonctions dans la cellule ci-dessous (elle contient le code fautif) : la vérification calcule c), la liste `[total_log_prob([0.5, 0.4]), log_of_sum(2, 3), grow(100, 0.05, 4, 6)]`.

Dans ta copie : une probabilité nulle dans un produit le rend nul. Que devient sa log-probabilité, et pourquoi un modèle qui donne la probabilité 0 à ce qui arrive vraiment est-il infiniment mauvais ?

In [ ]:
answer_R6_21a = ...  # a) letters, as text
answer_R6_21b = ...  # b) the name of an error, as text


# correct the three functions below
def total_log_prob(probabilities):
    """ln of the product of independent probabilities, computed as a sum of logarithms."""
    total = 1                                                     # (A)
    for p in probabilities:                                       # (B)
        total = total + math.log(p)                               # (C)
    return total                                                  # (D)


def log_of_sum(a, b):
    """ln(a + b), for a > 0 and b > 0."""
    return math.log(a) + math.log(b)                              # (E)


def grow(start, rate, t1, t2):
    """start × e^(rate × t1), then that value × e^(rate × t2): the value after t1 + t2 of continuous growth."""
    return start * (math.exp(rate * t1) + math.exp(rate * t2))    # (F)

In [ ]:
for letter, answer in zip("ab", [answer_R6_21a, answer_R6_21b]):
    wbm.check(f"R6.21{letter}", answer)
if answer_R6_21a is ...:
    print("⏳ Ex R6.21c : trouve d'abord les lignes fausses (a), puis corrige les trois fonctions.")
else:
    with checked("R6.21c"):
        wbm.check("R6.21c", [total_log_prob([0.5, 0.4]), log_of_sum(2, 3), grow(100, 0.05, 4, 6)], computed=True)

### Ex R6.22 — Lire la documentation de `numpy.log`, `log2`, `log10`, `log1p` 🛠️ ★ ⏱️ 15 min
**Objectif :** lire la documentation d'une fonction de NumPy pour savoir ce qu'elle calcule, sur quelles entrées, et pourquoi une variante existe.  
**Prérequis :** fiche §3 et §4 · **Parcours :** C

Une documentation dit ce qu'une fonction calcule, ce qu'elle rend pour les entrées interdites et à quoi servent ses variantes. Lis celle de `np.log1p` (la cellule en affiche le début ; `help(np.log1p)` ou la [page en ligne](https://numpy.org/doc/stable/reference/generated/numpy.log1p.html) donnent tout), puis celles de `np.log`, `np.log2` et `np.log10`. Complète la cellule avec tes expériences.

Dans ta copie :
1. Que calcule `np.log1p(x)` ? Pour quels $x$ est-il plus précis que `np.log(1 + x)`, et pourquoi (pense à ce que devient `1 + 1e-12` en flottant, R1) ?
2. Que rendent `np.log(0.0)` et `np.log(-1.0)` ? Et `math.log` dans les mêmes cas (R6.21) ? Lequel des deux comportements préfères-tu pour détecter une erreur, et lequel pour calculer sur tout un tableau ?
3. Quelle fonction de NumPy calcule $\log_{2}$, laquelle $\log_{10}$, et laquelle $\ln$ ? Et `np.log` accepte-t-il une base en second argument, comme `math.log` ?

In [ ]:
# 1. The first lines of the documentation of np.log1p (np.info shows the full text; or help(np.log1p))
print(np.log1p.__doc__[:700])

# 2. Your experiments: np.log(1 + 1e-12) against np.log1p(1e-12); np.log2(8.0) and np.log10(0.001);
#    np.log(0.0) and np.log(-1.0) (inside `with np.errstate(all="ignore"):` to silence the alerts)

## ✅ Bilan

**Auto-évaluation** (note-toi de 0 à 3 dans `mon_travail/suivi/auto_evaluation.md`) :
1. Simplifies-tu une expression avec exp et ln, et résous-tu $e^{x} = c$, $a^{n} > c$ en surveillant le domaine et le sens des inégalités ?
2. Passes-tu d'une base de logarithme à une autre, et comptes-tu en bits et en nats ?
3. Sais-tu comparer des croissances, lire une échelle logarithmique, et passer d'une probabilité à sa cote et à son logit ?

**La suite** : R7, *Limites et dérivation*. Dans workbookIA, continue le chapitre 0A au rythme de ton tableau de bord.